# Delete Empty / Dummy Parcels

This notebook cleans `PREPARE_GDB` using this rule:

```text
Shape_Length = 8000
AND Shape_Area = 4000000
AND PARCEL_TYPE = 99
→ DELETE polygon
```

Workflow:

```text
1. Walk every feature class in PREPARE_GDB
2. Delete matching polygons
3. Repair Geometry on feature classes where deletion occurred
4. Delete feature classes with zero remaining polygons
5. Write 03_<ACQUIRED>-Delete_Empty.xlsx
```

The notebook modifies `PREPARE_GDB` in place. `RepairGeometry(..., "KEEP_NULL", "ESRI")` is used so it remains compatible with ArcGIS Pro Basic.

In [ ]:
from pathlib import Path
from collections import Counter
import math
import os
import traceback
import uuid

try:
    import arcpy
except ImportError as exc:
    raise RuntimeError("Run this notebook with the ArcGIS Pro Python environment.") from exc

arcpy.env.overwriteOutput = True

ACQUIRED = "2022-05"
BASE_DIR = Path(r"D:\A02-Projects\Refine")
PREPARE_GDB = BASE_DIR / "01_acquired" / f"A{ACQUIRED}.gdb"
REPORT_DIR = BASE_DIR / "05_reports" / ACQUIRED
EMPTY_REPORT = REPORT_DIR / f"03_{ACQUIRED}-Delete_Empty.xlsx"
REPORT_GDB = REPORT_DIR / f"_delete_empty_report_{ACQUIRED.replace('-', '_')}.gdb"

TARGET_SHAPE_LENGTH = 8000.0
TARGET_SHAPE_AREA = 4_000_000.0
TARGET_PARCEL_TYPE = 99

# Tiny tolerances only protect against floating-point representation noise.
LENGTH_ABS_TOL = 0.001
AREA_ABS_TOL = 0.01

print("ACQUIRED    :", ACQUIRED)
print("PREPARE_GDB:", PREPARE_GDB)
print("REPORT     :", EMPTY_REPORT)


## Helpers

Feature fields are resolved case-insensitively. If a required field is missing, the dummy-polygon deletion is skipped for that feature class and the issue is reported. Empty feature classes are still deleted.

In [ ]:
def field_map(dataset):
    return {f.name.upper(): f for f in arcpy.ListFields(str(dataset))}

def count_rows(dataset):
    return int(arcpy.management.GetCount(str(dataset))[0])

def create_file_gdb(path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    arcpy.management.CreateFileGDB(str(path.parent), path.name)

def setup():
    if not arcpy.Exists(str(PREPARE_GDB)):
        raise FileNotFoundError(f"PREPARE_GDB does not exist: {PREPARE_GDB}")
    REPORT_DIR.mkdir(parents=True, exist_ok=True)
    if arcpy.Exists(str(REPORT_GDB)):
        arcpy.management.Delete(str(REPORT_GDB))
    create_file_gdb(REPORT_GDB)

def list_feature_classes():
    old = arcpy.env.workspace
    try:
        arcpy.env.workspace = str(PREPARE_GDB)
        return sorted(arcpy.ListFeatureClasses() or [])
    finally:
        arcpy.env.workspace = old

def is_target_number(value, target, abs_tol):
    if value is None:
        return False
    try:
        number = float(value)
    except (TypeError, ValueError):
        return False
    return math.isclose(number, target, rel_tol=0.0, abs_tol=abs_tol)

def parcel_type_is_99(value):
    if value is None:
        return False
    try:
        return int(value) == TARGET_PARCEL_TYPE
    except (TypeError, ValueError):
        return str(value).strip() == str(TARGET_PARCEL_TYPE)

def delete_dummy_polygons(feature_class):
    fields = field_map(feature_class)
    length_field = fields.get("SHAPE_LENGTH")
    area_field = fields.get("SHAPE_AREA")
    parcel_type_field = fields.get("PARCEL_TYPE")

    missing = []
    if length_field is None: missing.append("Shape_Length")
    if area_field is None: missing.append("Shape_Area")
    if parcel_type_field is None: missing.append("PARCEL_TYPE")

    if missing:
        return {
            "deleted_count": 0,
            "status": "SKIPPED_MISSING_FIELDS",
            "issue": "Missing required field(s): " + ", ".join(missing),
        }

    deleted = 0
    with arcpy.da.UpdateCursor(
        str(feature_class),
        [length_field.name, area_field.name, parcel_type_field.name],
    ) as cursor:
        for row in cursor:
            match = (
                is_target_number(row[0], TARGET_SHAPE_LENGTH, LENGTH_ABS_TOL)
                and is_target_number(row[1], TARGET_SHAPE_AREA, AREA_ABS_TOL)
                and parcel_type_is_99(row[2])
            )
            if match:
                cursor.deleteRow()
                deleted += 1

    return {
        "deleted_count": deleted,
        "status": "DELETED" if deleted else "NO_MATCH",
        "issue": None,
    }

def repair_affected_feature_class(feature_class):
    arcpy.management.RepairGeometry(str(feature_class), "KEEP_NULL", "ESRI")


## Main processing

Pass 1 deletes matching polygons from every feature class. Pass 2 repairs only feature classes where at least one polygon was deleted. Pass 3 deletes any feature class with zero remaining polygons.

In [ ]:
def run_delete_empty():
    setup()
    feature_classes = list_feature_classes()
    print(f"Discovered {len(feature_classes):,} feature class(es).")

    results = []
    affected = set()

    # PASS 1 — delete matching polygons
    for index, fc_name in enumerate(feature_classes, start=1):
        fc = PREPARE_GDB / fc_name
        print(f"\n[{index}/{len(feature_classes)}] {fc_name}")

        before = count_rows(fc)
        record = {
            "FEATURE_CLASS": fc_name,
            "COUNT_BEFORE": before,
            "DUMMY_DELETED": 0,
            "COUNT_AFTER_DUMMY_DELETE": before,
            "GEOMETRY_REPAIRED": 0,
            "COUNT_AFTER_REPAIR": before,
            "FEATURE_CLASS_DELETED": 0,
            "COUNT_FINAL": before,
            "STATUS": None,
            "ISSUE": None,
        }

        try:
            result = delete_dummy_polygons(fc)
            deleted = int(result["deleted_count"])
            after_delete = count_rows(fc)

            record["DUMMY_DELETED"] = deleted
            record["COUNT_AFTER_DUMMY_DELETE"] = after_delete
            record["COUNT_AFTER_REPAIR"] = after_delete
            record["COUNT_FINAL"] = after_delete
            record["STATUS"] = result["status"]
            record["ISSUE"] = result["issue"]

            if deleted > 0:
                affected.add(fc_name)

            expected = before - deleted
            if expected != after_delete:
                raise RuntimeError(
                    f"Count reconciliation failed: before={before}, deleted={deleted}, "
                    f"expected_after={expected}, actual_after={after_delete}"
                )

            print(f"  before  : {before:,}")
            print(f"  deleted : {deleted:,}")
            print(f"  remain  : {after_delete:,}")

        except Exception as exc:
            record["STATUS"] = "ERROR"
            record["ISSUE"] = f"{type(exc).__name__}: {exc}"
            print("  ERROR:", record["ISSUE"])
            traceback.print_exc()

        results.append(record)

    result_by_fc = {r["FEATURE_CLASS"]: r for r in results}

    # PASS 2 — repair geometry on affected FCs
    print("\nPASS 2 — Repair Geometry on affected feature classes")
    for fc_name in sorted(affected):
        fc = PREPARE_GDB / fc_name
        record = result_by_fc[fc_name]
        if not arcpy.Exists(str(fc)):
            continue
        try:
            print(f"  Repair Geometry: {fc_name}")
            repair_affected_feature_class(fc)
            record["GEOMETRY_REPAIRED"] = 1
            after_repair = count_rows(fc)
            record["COUNT_AFTER_REPAIR"] = after_repair
            record["COUNT_FINAL"] = after_repair
            if record["STATUS"] != "ERROR":
                record["STATUS"] = "DELETED_AND_REPAIRED"
        except Exception as exc:
            record["STATUS"] = "ERROR_REPAIR"
            record["ISSUE"] = f"{type(exc).__name__}: {exc}"
            print(f"  ERROR repairing {fc_name}: {record['ISSUE']}")
            traceback.print_exc()

    # PASS 3 — delete empty FCs
    print("\nPASS 3 — Delete empty feature classes")
    for fc_name in list_feature_classes():
        fc = PREPARE_GDB / fc_name
        final_count = count_rows(fc)
        record = result_by_fc.get(fc_name)
        if record is None:
            record = {
                "FEATURE_CLASS": fc_name,
                "COUNT_BEFORE": final_count,
                "DUMMY_DELETED": 0,
                "COUNT_AFTER_DUMMY_DELETE": final_count,
                "GEOMETRY_REPAIRED": 0,
                "COUNT_AFTER_REPAIR": final_count,
                "FEATURE_CLASS_DELETED": 0,
                "COUNT_FINAL": final_count,
                "STATUS": "DISCOVERED_LATE",
                "ISSUE": None,
            }
            results.append(record)
            result_by_fc[fc_name] = record

        record["COUNT_FINAL"] = final_count
        if final_count == 0:
            try:
                print(f"  Delete empty FC: {fc_name}")
                arcpy.management.Delete(str(fc))
                record["FEATURE_CLASS_DELETED"] = 1
                record["COUNT_FINAL"] = 0
                if record["STATUS"] not in {"ERROR", "ERROR_REPAIR"}:
                    record["STATUS"] = "EMPTY_FC_DELETED"
            except Exception as exc:
                record["STATUS"] = "ERROR_DELETE_EMPTY_FC"
                record["ISSUE"] = f"{type(exc).__name__}: {exc}"
                print(f"  ERROR deleting empty FC {fc_name}: {record['ISSUE']}")
                traceback.print_exc()

    return results

RESULTS = run_delete_empty()

print("\nProcessing finished.")
print(f"Feature classes reported: {len(RESULTS):,}")
print(f"Dummy polygons deleted   : {sum(int(r['DUMMY_DELETED'] or 0) for r in RESULTS):,}")
print(f"Empty FCs deleted        : {sum(int(r['FEATURE_CLASS_DELETED'] or 0) for r in RESULTS):,}")


## Report

The workbook contains `SUMMARY` and `FEATURE_CLASSES` sheets. The latter records before/after counts, dummy-polygon deletions, geometry repair, feature-class deletion, status, and any issue.

In [ ]:
FEATURE_CLASS_SCHEMA = [
    ("FEATURE_CLASS", "TEXT", 150),
    ("COUNT_BEFORE", "LONG", None),
    ("DUMMY_DELETED", "LONG", None),
    ("COUNT_AFTER_DUMMY_DELETE", "LONG", None),
    ("GEOMETRY_REPAIRED", "SHORT", None),
    ("COUNT_AFTER_REPAIR", "LONG", None),
    ("FEATURE_CLASS_DELETED", "SHORT", None),
    ("COUNT_FINAL", "LONG", None),
    ("STATUS", "TEXT", 60),
    ("ISSUE", "TEXT", 2000),
]

SUMMARY_SCHEMA = [
    ("METRIC", "TEXT", 100),
    ("VALUE", "LONG", None),
]

def create_report_table(table_name, schema, rows):
    table = REPORT_GDB / table_name
    if arcpy.Exists(str(table)):
        arcpy.management.Delete(str(table))
    arcpy.management.CreateTable(str(REPORT_GDB), table_name)

    for name, field_type, length in schema:
        kwargs = {}
        if field_type == "TEXT":
            kwargs["field_length"] = int(length)
        arcpy.management.AddField(str(table), name, field_type, **kwargs)

    fields = [name for name, _, _ in schema]
    if rows:
        with arcpy.da.InsertCursor(str(table), fields) as cursor:
            for record in rows:
                cursor.insertRow([record.get(field) for field in fields])
    return table

def build_summary_rows(results):
    status_counts = Counter(r["STATUS"] for r in results)
    rows = [
        {"METRIC": "FEATURE_CLASSES_REPORTED", "VALUE": len(results)},
        {"METRIC": "TOTAL_COUNT_BEFORE", "VALUE": sum(int(r["COUNT_BEFORE"] or 0) for r in results)},
        {"METRIC": "TOTAL_DUMMY_POLYGONS_DELETED", "VALUE": sum(int(r["DUMMY_DELETED"] or 0) for r in results)},
        {"METRIC": "FEATURE_CLASSES_REPAIRED", "VALUE": sum(int(r["GEOMETRY_REPAIRED"] or 0) for r in results)},
        {"METRIC": "EMPTY_FEATURE_CLASSES_DELETED", "VALUE": sum(int(r["FEATURE_CLASS_DELETED"] or 0) for r in results)},
        {"METRIC": "TOTAL_FINAL_POLYGONS", "VALUE": sum(int(r["COUNT_FINAL"] or 0) for r in results if not int(r["FEATURE_CLASS_DELETED"] or 0))},
    ]
    for status, count in sorted(status_counts.items(), key=lambda x: str(x[0])):
        rows.append({"METRIC": f"STATUS_{status}", "VALUE": int(count)})
    return rows

def write_report(results):
    summary_table = create_report_table("SUMMARY", SUMMARY_SCHEMA, build_summary_rows(results))
    detail_table = create_report_table("FEATURE_CLASSES", FEATURE_CLASS_SCHEMA, results)

    temp_report = REPORT_DIR / f"_tmp_Delete_Empty_{uuid.uuid4().hex[:8]}.xlsx"
    arcpy.conversion.TableToExcel([str(summary_table), str(detail_table)], str(temp_report))

    try:
        if EMPTY_REPORT.exists():
            EMPTY_REPORT.unlink()
        os.replace(temp_report, EMPTY_REPORT)
        print(f"\nReport written: {EMPTY_REPORT}")
    except PermissionError:
        print("\nWARNING: target Excel report appears to be open/locked.")
        print("New report was left at:")
        print(temp_report)

write_report(RESULTS)


## Expected output

```text
D:\A02-Projects\Refine\05_reports\2022-05\03_2022-05-Delete_Empty.xlsx
```

The notebook is safe to rerun after a completed run: deleted polygons and deleted empty feature classes are already gone and therefore will not be processed again.

This artifact is syntax-validated but cannot be executed here because ArcPy and your local geodatabases are unavailable.